In [ ]:
# ==========================================
# 1. THƯ VIỆN & CẤU HÌNH BAN ĐẦU
# ==========================================
import datetime
import hashlib
import json
import logging
import os
import platform
import sys
import time
from collections import Counter
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Kaggle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    top_k_accuracy_score,
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
import yaml

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ==========================================
# 2. CẤU HÌNH & CẤU TRÚC THƯ MỤC
#    (Tuân thủ train_request.md)
# ==========================================

# --- RUN CONFIGURATION ---
RUN_ID = "attr_head_v1"
MODULE_NAME = "attribute_resnet18_head_tune"
RUNNER = "NguyenQuocBao"  # Tên người chạy
SEED = 42

# --- HYPERPARAMETERS ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3
OPTIMIZER_NAME = "adamw"
SCHEDULER_NAME = "reduce_lr_on_plateau"
WEIGHT_DECAY = 1e-4
COLOR_LOSS_WEIGHT = 2.0

# --- REPRODUCIBILITY ---
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- DATASET PATHS ---
candidate_base_dirs = [
    Path("/kaggle/input/rximage-new/rximage"),
    Path("c:/ML_DL_Project/Data_rximage_kaggle/rximage"),
    Path("c:/ML_DL_Project/Data/rximage"),
    Path("Data/rximage"),
]

BASE_DIR = None
for p in candidate_base_dirs:
    if p.exists() and (p / 'combined').exists():
        BASE_DIR = p
        break

if BASE_DIR is None:
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for sub in kaggle_input.rglob("combined"):
            if sub.is_dir():
                BASE_DIR = sub.parent
                break

if BASE_DIR is None:
    BASE_DIR = Path("/kaggle/input/rximage_processed/Data/rximage")

COMBINED_DIR = BASE_DIR / "combined"
IMG_DIR = BASE_DIR / "image_all"

TRAIN_CSV = COMBINED_DIR / "train_combined_crop.csv"
VAL_CSV = COMBINED_DIR / "val_combined_crop.csv"
TEST_CSV = COMBINED_DIR / "test_combined_crop.csv"

# --- OUTPUT DIRECTORIES (theo train_request.md §1) ---
EXPERIMENT_DIR = Path(f"/kaggle/working/experiments/{MODULE_NAME}")
SUBDIRS = ["checkpoints", "logs", "metrics", "plots", "predictions"]

PATHS = {}
for sub in SUBDIRS:
    path = EXPERIMENT_DIR / sub
    path.mkdir(parents=True, exist_ok=True)
    PATHS[sub] = path

# Predictions subfolder (theo §5.3)
PRED_DIR = PATHS["predictions"] / RUN_ID
for folder in ["correct_samples", "wrong_shape", "wrong_color", "low_confidence"]:
    (PRED_DIR / folder).mkdir(parents=True, exist_ok=True)

# --- LOGGER ---
def setup_logger(name, log_file):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    if logger.hasHandlers():
        logger.handlers.clear()
    handler = logging.FileHandler(log_file)
    handler.setFormatter(logging.Formatter("%(asctime)s - %(message)s"))
    logger.addHandler(handler)
    return logger

logger = setup_logger("heads_finetune", PATHS["logs"] / f"{RUN_ID}_training.log")

print(f"✓ Run ID: {RUN_ID}")
print(f"✓ Module: {MODULE_NAME}")
print(f"✓ Experiment Dir: {EXPERIMENT_DIR}")
print(f"✓ Dataset Dir: {BASE_DIR}")
print(f"✓ Device: {DEVICE}")
print(f"✓ Seed: {SEED}")

In [ ]:
# ==========================================
# 3. DATASET CLASS
# ==========================================
MISSING_FILES = {
    "63459-0502-30_RXNAVIMAGE10_8641C37E_1.jpg",
    "63459-0502-30_RXNAVIMAGE10_8641C37E_2.jpg",
    "63459-0504-30_RXNAVIMAGE10_8841C43E_1.jpg",
    "63459-0504-30_RXNAVIMAGE10_8841C43E_2.jpg",
    "63459-0506-30_RXNAVIMAGE10_8941C4CE_1.jpg",
    "63459-0506-30_RXNAVIMAGE10_8941C4CE_2.jpg",
    "63459-0508-30_RXNAVIMAGE10_8941C4FE_1.jpg",
    "63459-0508-30_RXNAVIMAGE10_8941C4FE_2.jpg",
    "63459-0512-30_RXNAVIMAGE10_8A41C55E_1.jpg",
    "63459-0512-30_RXNAVIMAGE10_8A41C55E_2.jpg",
    "63459-0516-30_RXNAVIMAGE10_8B41C5BE_1.jpg",
    "63459-0516-30_RXNAVIMAGE10_8B41C5BE_2.jpg",
}

class RxImageDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, shape_encoder=None, mlb_color=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = Path(img_dir)
        self.transform = transform

        # Filter out 12 missing images from CSV automatically
        filename_col = "rxnavImageFileName" if "rxnavImageFileName" in self.df.columns else "filename"
        if filename_col in self.df.columns:
            before_len = len(self.df)
            self.df = self.df[~self.df[filename_col].isin(MISSING_FILES)].reset_index(drop=True)
            if len(self.df) < before_len:
                print(f"  [INFO] Auto-filtered {before_len - len(self.df)} missing image rows from {Path(csv_file).name}")

        # 1. Shape encoding
        has_shape_label = "shape_label" in self.df.columns
        has_shape_name = "shape" in self.df.columns

        if has_shape_label:
            self.shape_labels = self.df["shape_label"].values
            self.shape_encoder = shape_encoder
            
            # Build a simple dict to map label -> name if both columns exist
            if has_shape_name and shape_encoder is None:
                mapping = self.df.dropna(subset=["shape", "shape_label"]).drop_duplicates(subset=["shape_label"])
                self.shape_encoder_dict = dict(zip(mapping["shape_label"], mapping["shape"]))
            else:
                self.shape_encoder_dict = getattr(shape_encoder, "shape_encoder_dict", None) if shape_encoder else None
        elif has_shape_name:
            from sklearn.preprocessing import LabelEncoder
            if shape_encoder is None:
                self.shape_encoder = LabelEncoder()
                self.shape_labels = self.shape_encoder.fit_transform(self.df["shape"].fillna("UNKNOWN").astype(str))
                self.shape_encoder_dict = {i: name for i, name in enumerate(self.shape_encoder.classes_)}
            else:
                self.shape_encoder = shape_encoder
                self.shape_labels = self.shape_encoder.transform(self.df["shape"].fillna("UNKNOWN").astype(str))
                self.shape_encoder_dict = getattr(shape_encoder, "shape_encoder_dict", None)
        else:
            raise KeyError("CSV missing both 'shape_label' and 'shape' columns.")

        # 2. Color encoding (Multi-label)
        self.color_cols = [c for c in self.df.columns if c.startswith("color_")]
        if len(self.color_cols) > 0:
            self.color_labels = self.df[self.color_cols].values.astype(np.float32)
        elif "color" in self.df.columns:
            from sklearn.preprocessing import MultiLabelBinarizer
            def parse_colors(color_str):
                if not color_str or pd.isna(color_str):
                    return ["unknown"]
                colors = str(color_str).replace(";", " ").replace("/", " ").replace(",", " ").split()
                return [c.strip().lower() for c in colors if c.strip()]

            color_series = self.df["color"].apply(parse_colors)
            if mlb_color is None:
                self.mlb_color = MultiLabelBinarizer()
                color_bin = self.mlb_color.fit_transform(color_series)
            else:
                self.mlb_color = mlb_color
                color_bin = self.mlb_color.transform(color_series)
            self.color_labels = color_bin.astype(np.float32)
            self.color_cols = [f"color_{c}" for c in getattr(self.mlb_color, "classes_", [])]
        else:
            raise KeyError("CSV missing both 'color_*' columns and 'color' column.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = str(row.get("rxnavImageFileName", row.get("filename", ""))).strip()
        img_path = self.img_dir / filename

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        shape_target = torch.tensor(self.shape_labels[idx], dtype=torch.long)
        color_target = torch.tensor(self.color_labels[idx], dtype=torch.float32)

        return image, shape_target, color_target


In [ ]:
# ==========================================
# 4. TRANSFORMS & DATALOADERS
# ==========================================
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    "val": transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
}

train_dataset = RxImageDataset(TRAIN_CSV, IMG_DIR, transform=data_transforms["train"])
shape_encoder = getattr(train_dataset, 'shape_encoder', None)
mlb_color = getattr(train_dataset, 'mlb_color', None)
val_dataset = RxImageDataset(VAL_CSV, IMG_DIR, transform=data_transforms["val"], shape_encoder=shape_encoder, mlb_color=mlb_color)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

NUM_SHAPE_CLASSES = int(max(train_dataset.shape_labels.max(), val_dataset.shape_labels.max())) + 1
NUM_COLOR_CLASSES = len(train_dataset.color_cols)

# --- Label mapping (bắt buộc cho attribute model, §3.2) ---
# Get real shape names from CSV dictionary mapping or fallback
shape_encoder_dict = getattr(train_dataset, "shape_encoder_dict", None)
if shape_encoder_dict is not None:
    shape_class_names = [shape_encoder_dict.get(i, f"shape_class_{i}") for i in range(NUM_SHAPE_CLASSES)]
elif shape_encoder is not None and hasattr(shape_encoder, 'classes_'):
    shape_class_names = list(shape_encoder.classes_)
elif "shape" in train_dataset.df.columns:
    shape_class_names = sorted(train_dataset.df["shape"].dropna().unique().tolist())
else:
    shape_class_names = [f"shape_class_{i}" for i in range(NUM_SHAPE_CLASSES)]
color_class_names = train_dataset.color_cols

label_mapping = {
    "shape": {int(i): name for i, name in enumerate(shape_class_names)},
    "color": color_class_names,
}

label_mapping_path = PATHS["logs"] / f"{RUN_ID}_label_mapping.json"
with open(label_mapping_path, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2, ensure_ascii=False)

# --- Class distribution ---
train_shape_dist = dict(Counter(train_dataset.shape_labels.tolist()))
train_shape_dist_named = {shape_class_names[int(k)]: v for k, v in train_shape_dist.items()}

print(f"✓ Số lớp Shape: {NUM_SHAPE_CLASSES} | Số nhãn Color: {NUM_COLOR_CLASSES}")
print(f"✓ Shape classes: {shape_class_names}")
print(f"✓ Color classes: {color_class_names}")
print(f"✓ Label mapping saved: {label_mapping_path}")
print(f"✓ Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# --- Filter missing images (12 ảnh thiếu từ manufacturer 63459) ---
MISSING_FILES = {
    "63459-0502-30_RXNAVIMAGE10_8641C37E_1.jpg",
    "63459-0502-30_RXNAVIMAGE10_8641C37E_2.jpg",
    "63459-0504-30_RXNAVIMAGE10_8841C43E_1.jpg",
    "63459-0504-30_RXNAVIMAGE10_8841C43E_2.jpg",
    "63459-0506-30_RXNAVIMAGE10_8941C4CE_1.jpg",
    "63459-0506-30_RXNAVIMAGE10_8941C4CE_2.jpg",
    "63459-0508-30_RXNAVIMAGE10_8941C4FE_1.jpg",
    "63459-0508-30_RXNAVIMAGE10_8941C4FE_2.jpg",
    "63459-0512-30_RXNAVIMAGE10_8A41C55E_1.jpg",
    "63459-0512-30_RXNAVIMAGE10_8A41C55E_2.jpg",
    "63459-0516-30_RXNAVIMAGE10_8B41C5BE_1.jpg",
    "63459-0516-30_RXNAVIMAGE10_8B41C5BE_2.jpg",
}

filename_col = "rxnavImageFileName" if "rxnavImageFileName" in train_dataset.df.columns else "filename"
for ds_name, ds in [("train", train_dataset), ("val", val_dataset)]:
    before = len(ds.df)
    mask = ~ds.df[filename_col].isin(MISSING_FILES)
    ds.df = ds.df[mask].reset_index(drop=True)
    ds.shape_labels = ds.shape_labels[mask.values]
    ds.color_labels = ds.color_labels[mask.values]
    after = len(ds.df)
    if before != after:
        print(f"  ⚠️ Filtered {before - after} missing images from {ds_name} set")

# --- Loại cột color_BLACK (chỉ 2 mẫu, không đủ để train) ---
BLACK_COL = "color_BLACK"
if BLACK_COL in train_dataset.color_cols:
    black_idx = train_dataset.color_cols.index(BLACK_COL)
    # Remove from train
    train_dataset.color_cols = [c for c in train_dataset.color_cols if c != BLACK_COL]
    train_dataset.color_labels = np.delete(train_dataset.color_labels, black_idx, axis=1)
    # Remove from val
    val_dataset.color_cols = [c for c in val_dataset.color_cols if c != BLACK_COL]
    val_dataset.color_labels = np.delete(val_dataset.color_labels, black_idx, axis=1)
    NUM_COLOR_CLASSES = len(train_dataset.color_cols)
    print(f"✓ Removed {BLACK_COL} (only 2 samples). Color classes: {NUM_COLOR_CLASSES}")

# Update color class names for label mapping
color_class_names = train_dataset.color_cols

# Update label mapping without BLACK
label_mapping = {
    "shape": {int(i): name for i, name in enumerate(shape_class_names)},
    "color": color_class_names,
}

label_mapping_path = PATHS["logs"] / f"{RUN_ID}_label_mapping.json"
with open(label_mapping_path, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2, ensure_ascii=False)

print(f"✓ Updated label mapping: {len(shape_class_names)} shapes, {NUM_COLOR_CLASSES} colors")

In [ ]:
# ==========================================
# 7. MÔ HÌNH: FREEZE BACKBONE + HEADS
# ==========================================
class MultiTaskResNet18_HeadsFinetune(nn.Module):
    def __init__(self, num_shape_classes, num_color_classes, pretrained=True):
        super(MultiTaskResNet18_HeadsFinetune, self).__init__()

        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        # FREEZE BACKBONE
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Shape Head
        self.fc_shape = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_ftrs, num_shape_classes),
        )

        # Color Head (enhanced)
        self.fc_color = nn.Sequential(
            nn.Linear(num_ftrs, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_color_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        shape_out = self.fc_shape(features)
        color_out = self.fc_color(features)
        return shape_out, color_out


model = MultiTaskResNet18_HeadsFinetune(
    num_shape_classes=NUM_SHAPE_CLASSES,
    num_color_classes=NUM_COLOR_CLASSES,
    pretrained=True,
).to(DEVICE)

# Optimizer: chỉ train heads
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5)

# Class weights for imbalanced shape classes (96.6% in top 3)
from collections import Counter as _Counter
_shape_counts = _Counter(train_dataset.shape_labels.tolist())
_total_shape = sum(_shape_counts.values())
_n_classes = len(_shape_counts)
shape_weights = torch.zeros(_n_classes)
for _cls_idx, _count in _shape_counts.items():
    shape_weights[_cls_idx] = _total_shape / (_n_classes * _count)
shape_weights = shape_weights.to(DEVICE)
criterion_shape = nn.CrossEntropyLoss(weight=shape_weights)
print(f"✓ Shape class weights applied ({_n_classes} classes)")
criterion_color = nn.BCEWithLogitsLoss()

total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,} | Trainable (Heads): {total_trainable:,} ({total_trainable/total_params*100:.2f}%)")

In [ ]:
# ==========================================
# 11. ĐÁNH GIÁ TRÊN TEST SET
# ==========================================
test_dataset = RxImageDataset(
    TEST_CSV, IMG_DIR, transform=data_transforms["val"],
    shape_encoder=shape_encoder, mlb_color=mlb_color,
)
# Align color columns with train_dataset (e.g., removing color_BLACK)
if len(test_dataset.color_cols) != len(train_dataset.color_cols):
    keep_indices = [test_dataset.color_cols.index(c) for c in train_dataset.color_cols if c in test_dataset.color_cols]
    test_dataset.color_cols = [test_dataset.color_cols[i] for i in keep_indices]
    test_dataset.color_labels = test_dataset.color_labels[:, keep_indices]

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Load best checkpoint
best_ckpt_path = PATHS["checkpoints"] / f"{RUN_ID}_best.pt"
if not best_ckpt_path.exists():
    best_ckpt_path = Path(f"/kaggle/working/{RUN_ID}_best.pt")

best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()

print(f"Loaded best checkpoint from epoch {best_ckpt['epoch']}")

# --- Collect predictions ---
all_shape_preds, all_shape_targets = [], []
all_color_preds, all_color_targets = [], []
all_shape_probs = []
all_color_probs = []

with torch.no_grad():
    for images, s_targets, c_targets in test_loader:
        images = images.to(DEVICE)
        s_outputs, c_outputs = model(images)

        s_probs = torch.softmax(s_outputs, dim=1)
        _, s_preds = torch.max(s_outputs, 1)
        all_shape_preds.extend(s_preds.cpu().numpy())
        all_shape_targets.extend(s_targets.numpy())
        all_shape_probs.append(s_probs.cpu().numpy())

        c_probs = torch.sigmoid(c_outputs)
        c_preds = (c_probs > 0.5).int()
        all_color_preds.append(c_preds.cpu().numpy())
        all_color_targets.append(c_targets.numpy())
        all_color_probs.append(c_probs.cpu().numpy())

shape_preds = np.array(all_shape_preds)
shape_targets = np.array(all_shape_targets)
shape_probs = np.vstack(all_shape_probs)
color_preds = np.vstack(all_color_preds)
color_targets = np.vstack(all_color_targets)
color_probs = np.vstack(all_color_probs)

print(f"Test samples: {len(shape_preds)}")
print(f"Shape accuracy: {np.mean(shape_preds == shape_targets):.4f}")
print(f"Shape F1 (macro): {f1_score(shape_targets, shape_preds, average='macro', zero_division=0):.4f}")
print(f"Color F1 (macro): {f1_score(color_targets, color_preds, average='macro', zero_division=0):.4f}")

In [ ]:
# ==========================================
# 12. LƯU VAL METRICS JSON (train_request.md §3.5)
# ==========================================
val_metrics = {
    "run_id": RUN_ID,
    "module": MODULE_NAME,
    "split": "val",
    "best_epoch": best_epoch,
    "best_checkpoint": str(PATHS["checkpoints"] / f"{RUN_ID}_best.pt"),
    "selection_metric": "overall_macro_f1",
    "metrics": {
        "shape_macro_f1": round(history["val_shape_f1"][best_epoch - 1], 4),
        "color_macro_f1": round(history["val_color_f1"][best_epoch - 1], 4),
        "dosage_form_macro_f1": None,
        "scoreline_macro_f1": None,
        "overall_macro_f1": round(best_val_f1, 4),
    },
    "label_mapping_file": str(label_mapping_path),
}

val_metrics_path = PATHS["metrics"] / f"{RUN_ID}_val_metrics.json"
with open(val_metrics_path, "w", encoding="utf-8") as f:
    json.dump(val_metrics, f, indent=2, ensure_ascii=False)

print(f"✓ Val metrics saved: {val_metrics_path}")
print(json.dumps(val_metrics, indent=2))

In [ ]:
# ==========================================
# 13. LƯU TEST METRICS JSON (train_request.md §3.6)
# ==========================================
shape_f1_test = float(f1_score(shape_targets, shape_preds, average="macro", zero_division=0))
color_f1_test = float(f1_score(color_targets, color_preds, average="macro", zero_division=0))
overall_f1_test = (shape_f1_test + color_f1_test) / 2.0

# Per-class metrics
shape_report = classification_report(
    shape_targets, shape_preds,
    labels=range(NUM_SHAPE_CLASSES),
    target_names=shape_class_names,
    output_dict=True, zero_division=0,
)
color_per_class = {}
for i, name in enumerate(color_class_names):
    col_t = color_targets[:, i]
    col_p = color_preds[:, i]
    color_per_class[name] = {
        "precision": round(float(precision_score(col_t, col_p, zero_division=0)), 4),
        "recall": round(float(recall_score(col_t, col_p, zero_division=0)), 4),
        "f1-score": round(float(f1_score(col_t, col_p, zero_division=0)), 4),
        "support": int(col_t.sum()),
    }

test_metrics = {
    "run_id": RUN_ID,
    "module": MODULE_NAME,
    "split": "test",
    "best_epoch": best_epoch,
    "best_checkpoint": str(PATHS["checkpoints"] / f"{RUN_ID}_best.pt"),
    "selection_metric": "overall_macro_f1",
    "metrics": {
        "shape_macro_f1": round(shape_f1_test, 4),
        "color_macro_f1": round(color_f1_test, 4),
        "dosage_form_macro_f1": None,
        "scoreline_macro_f1": None,
        "overall_macro_f1": round(overall_f1_test, 4),
        "shape_accuracy": round(float(np.mean(shape_preds == shape_targets)), 4),
        "shape_balanced_accuracy": round(float(balanced_accuracy_score(shape_targets, shape_preds)), 4),
        "color_precision_macro": round(float(precision_score(color_targets, color_preds, average="macro", zero_division=0)), 4),
        "color_recall_macro": round(float(recall_score(color_targets, color_preds, average="macro", zero_division=0)), 4),
    },
    "per_class_metrics": {
        "shape": {k: v for k, v in shape_report.items() if k not in ["accuracy", "macro avg", "weighted avg"]},
        "color": color_per_class,
    },
    "label_mapping_file": str(label_mapping_path),
}

test_metrics_path = PATHS["metrics"] / f"{RUN_ID}_test_metrics.json"
with open(test_metrics_path, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2, ensure_ascii=False)

print(f"✓ Test metrics saved: {test_metrics_path}")
print(f"  Shape F1: {shape_f1_test:.4f} | Color F1: {color_f1_test:.4f} | Overall: {overall_f1_test:.4f}")

In [ ]:
# ==========================================
# 14. VẼ CONFUSION MATRIX & PER-CLASS F1
#     (train_request.md §5.4)
# ==========================================

# --- 14a. Shape Confusion Matrix ---
cm = confusion_matrix(shape_targets, shape_preds)
plt.figure(figsize=(14, 11))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=shape_class_names, yticklabels=shape_class_names,
)
plt.title(f"Shape Confusion Matrix — {RUN_ID} (Test Set)", fontsize=14, fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(PATHS["plots"] / f"{RUN_ID}_shape_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {RUN_ID}_shape_confusion_matrix.png")

# --- 14b. Color per-class F1 ---
color_f1_per_class = [color_per_class[name]["f1-score"] for name in color_class_names]
color_support = [color_per_class[name]["support"] for name in color_class_names]

# Sort by F1
sorted_idx = np.argsort(color_f1_per_class)
sorted_names = [color_class_names[i] for i in sorted_idx]
sorted_f1 = [color_f1_per_class[i] for i in sorted_idx]
sorted_support = [color_support[i] for i in sorted_idx]

fig, ax = plt.subplots(figsize=(12, max(6, len(color_class_names) * 0.4)))
bars = ax.barh(range(len(sorted_names)), sorted_f1, color=plt.cm.RdYlGn(np.array(sorted_f1)))
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels([f"{n} (n={s})" for n, s in zip(sorted_names, sorted_support)])
ax.set_xlabel("F1 Score")
ax.set_title(f"Color Per-class F1 — {RUN_ID} (Test Set)", fontweight="bold")
ax.set_xlim(0, 1.05)
ax.axvline(x=color_f1_test, color='red', linestyle='--', label=f"Macro F1: {color_f1_test:.4f}")
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

for i, (f1_val, name) in enumerate(zip(sorted_f1, sorted_names)):
    ax.text(f1_val + 0.01, i, f"{f1_val:.3f}", va="center", fontsize=8)

plt.tight_layout()
plt.savefig(PATHS["plots"] / f"{RUN_ID}_color_f1_per_class.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {RUN_ID}_color_f1_per_class.png")

In [ ]:
# ==========================================
# 15. SUMMARY PLOT (train_request.md §3.7)
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f"Training Summary — {RUN_ID} ({MODULE_NAME})", fontsize=16, fontweight="bold")

# Loss curve
axes[0, 0].plot(history["epoch"], history["train_loss"], 'b-o', label="Train", markersize=3)
axes[0, 0].plot(history["epoch"], history["val_loss"], 'r-o', label="Val", markersize=3)
axes[0, 0].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5)
axes[0, 0].set_title("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# F1 curves
axes[0, 1].plot(history["epoch"], history["val_shape_f1"], 'r-o', label="Shape F1", markersize=3)
axes[0, 1].plot(history["epoch"], history["val_color_f1"], 'b-o', label="Color F1", markersize=3)
axes[0, 1].plot(history["epoch"], overall_f1_history, 'g-s', label="Overall F1", markersize=3)
axes[0, 1].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5)
axes[0, 1].set_title("Macro F1 (Val)")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Shape confusion matrix (small)
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)
cm_normalized = np.nan_to_num(cm_normalized)
sns.heatmap(
    cm_normalized, annot=False, cmap="Blues", ax=axes[1, 0],
    xticklabels=shape_class_names, yticklabels=shape_class_names,
    cbar_kws={'shrink': 0.8},
)
axes[1, 0].set_title("Shape Confusion (Normalized)")
axes[1, 0].tick_params(axis='x', rotation=45)

# Results table
axes[1, 1].axis('off')
table_data = [
    ["Run ID", RUN_ID],
    ["Strategy", "head_tune"],
    ["Epochs", f"{NUM_EPOCHS} (best: {best_epoch})"],
    ["LR", str(LEARNING_RATE)],
    ["Batch Size", str(BATCH_SIZE)],
    ["", ""],
    ["Val Shape F1", f"{history['val_shape_f1'][best_epoch-1]:.4f}"],
    ["Val Color F1", f"{history['val_color_f1'][best_epoch-1]:.4f}"],
    ["Val Overall F1", f"{best_val_f1:.4f}"],
    ["", ""],
    ["Test Shape F1", f"{shape_f1_test:.4f}"],
    ["Test Color F1", f"{color_f1_test:.4f}"],
    ["Test Overall F1", f"{overall_f1_test:.4f}"],
]
table = axes[1, 1].table(
    cellText=table_data, colLabels=["Metric", "Value"],
    loc='center', cellLoc='left',
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
axes[1, 1].set_title("Results Summary")

plt.tight_layout()
plt.savefig(PATHS["plots"] / f"{RUN_ID}_summary.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {RUN_ID}_summary.png")

In [ ]:
# ==========================================
# 16. LƯU PREDICTIONS & PHÂN LOẠI ERROR CASES
#     (train_request.md §5.3)
# ==========================================
test_df = pd.read_csv(TEST_CSV)
filenames_col = "rxnavImageFileName" if "rxnavImageFileName" in test_df.columns else "filename"
filenames = test_df[filenames_col].tolist()

# Build prediction records
predictions = []
for i in range(len(shape_preds)):
    shape_pred_idx = int(shape_preds[i])
    shape_true_idx = int(shape_targets[i])
    shape_conf = float(shape_probs[i, shape_pred_idx])

    # Color predictions
    color_pred_names = [color_class_names[j] for j in range(NUM_COLOR_CLASSES) if color_preds[i, j] == 1]
    color_true_names = [color_class_names[j] for j in range(NUM_COLOR_CLASSES) if color_targets[i, j] == 1]
    color_conf = float(np.mean([color_probs[i, j] for j in range(NUM_COLOR_CLASSES) if color_preds[i, j] == 1])) if color_pred_names else 0.0

    shape_correct = shape_pred_idx == shape_true_idx
    color_correct = np.array_equal(color_preds[i], color_targets[i])

    # Categorize
    if shape_correct and color_correct:
        category = "correct_samples"
    elif not shape_correct:
        category = "wrong_shape"
    elif not color_correct:
        category = "wrong_color"
    else:
        category = "wrong_shape"

    if shape_conf < 0.5 or color_conf < 0.5:
        category = "low_confidence"

    record = {
        "image_path": filenames[i] if i < len(filenames) else f"sample_{i}",
        "category": category,
        "ground_truth": {
            "shape": shape_class_names[shape_true_idx],
            "color": color_true_names,
        },
        "prediction": {
            "shape": {
                "label": shape_class_names[shape_pred_idx],
                "confidence": round(shape_conf, 4),
            },
            "color": {
                "labels": color_pred_names,
                "confidence": round(color_conf, 4),
            },
        },
        "correct": {
            "shape": shape_correct,
            "color": color_correct,
        },
    }
    predictions.append(record)

# Save per-category JSON samples (max 50 per category)
category_counts = Counter(p["category"] for p in predictions)
for cat in ["correct_samples", "wrong_shape", "wrong_color", "low_confidence"]:
    cat_preds = [p for p in predictions if p["category"] == cat][:50]
    cat_path = PRED_DIR / cat / "samples.json"
    with open(cat_path, "w", encoding="utf-8") as f:
        json.dump(cat_preds, f, indent=2, ensure_ascii=False)

# Save all predictions as CSV
pred_rows = []
for p in predictions:
    pred_rows.append({
        "image_path": p["image_path"],
        "category": p["category"],
        "true_shape": p["ground_truth"]["shape"],
        "pred_shape": p["prediction"]["shape"]["label"],
        "shape_confidence": p["prediction"]["shape"]["confidence"],
        "true_colors": "|".join(p["ground_truth"]["color"]),
        "pred_colors": "|".join(p["prediction"]["color"]["labels"]),
        "color_confidence": p["prediction"]["color"]["confidence"],
        "shape_correct": p["correct"]["shape"],
        "color_correct": p["correct"]["color"],
    })

pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv(PRED_DIR / "test_predictions_detailed.csv", index=False)

print(f"✓ Predictions saved to: {PRED_DIR}")
print(f"  Category distribution:")
for cat, count in category_counts.items():
    print(f"    {cat}: {count} ({count/len(predictions)*100:.1f}%)")

In [ ]:
# ==========================================
# 17. CHECKLIST CUỐI (train_request.md §9)
# ==========================================
print("\n" + "=" * 70)
print("CHECKLIST — train_request.md §9")
print("=" * 70)

checks = [
    ("train_log.csv", (PATHS["logs"] / f"{RUN_ID}_train_log.csv").exists()),
    ("config.yaml", (PATHS["logs"] / f"{RUN_ID}_config.yaml").exists()),
    ("dataset_manifest.json", (PATHS["logs"] / f"{RUN_ID}_dataset_manifest.json").exists()),
    ("val_metrics.json", (PATHS["metrics"] / f"{RUN_ID}_val_metrics.json").exists()),
    ("test_metrics.json", (PATHS["metrics"] / f"{RUN_ID}_test_metrics.json").exists()),
    ("loss_curve.png", (PATHS["plots"] / f"{RUN_ID}_loss_curve.png").exists()),
    ("metric_curve.png", (PATHS["plots"] / f"{RUN_ID}_metric_curve.png").exists()),
    ("shape_confusion_matrix.png", (PATHS["plots"] / f"{RUN_ID}_shape_confusion_matrix.png").exists()),
    ("color_f1_per_class.png", (PATHS["plots"] / f"{RUN_ID}_color_f1_per_class.png").exists()),
    ("summary.png", (PATHS["plots"] / f"{RUN_ID}_summary.png").exists()),
    ("best checkpoint", (PATHS["checkpoints"] / f"{RUN_ID}_best.pt").exists()),
    ("last checkpoint", (PATHS["checkpoints"] / f"{RUN_ID}_last.pt").exists()),
    ("runtime.txt", (PATHS["logs"] / f"{RUN_ID}_runtime.txt").exists()),
    ("label_mapping.json", (PATHS["logs"] / f"{RUN_ID}_label_mapping.json").exists()),
    ("predictions/correct_samples", (PRED_DIR / "correct_samples" / "samples.json").exists()),
    ("predictions/wrong_shape", (PRED_DIR / "wrong_shape" / "samples.json").exists()),
    ("predictions/wrong_color", (PRED_DIR / "wrong_color" / "samples.json").exists()),
    ("predictions/low_confidence", (PRED_DIR / "low_confidence" / "samples.json").exists()),
]

all_passed = True
for name, exists in checks:
    status = "✅" if exists else "❌"
    if not exists:
        all_passed = False
    print(f"  {status} {name}")

print("\n" + "=" * 70)
if all_passed:
    print("🎉 TẤT CẢ FILE ĐÃ ĐƯỢC TẠO THÀNH CÔNG!")
else:
    print("⚠️  CÓ FILE CHƯA ĐƯỢC TẠO — KIỂM TRA LẠI!")

print(f"\nOutput directory: {EXPERIMENT_DIR}")
print("\nĐể copy sang repo, chạy lệnh sau:")
print(f"  cp -r {EXPERIMENT_DIR}/* experiments/{MODULE_NAME}/")
print(f"  cp {PATHS['checkpoints']}/{RUN_ID}_best.pt models/{MODULE_NAME}/")